In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
import json 


import numpy as np 
import pandas as pd 
from pathlib import Path
from datasets import load_dataset
import subprocess

from huggingface_hub import hf_hub_download

d:\ai_project\FinGuide-AI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
PROJECT_ROOT = Path.cwd().parent
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

finqa_dataset_path = RAW_DATA_DIR / "FinQA" / "dataset"
FINQA_30K_DIR = RAW_DATA_DIR / "FinQA-30K"
CONVFINQA_DIR = RAW_DATA_DIR / "ConvFinQA"

print("FinQA:", finqa_dataset_path)
print("FinQA-30K:", FINQA_30K_DIR)
print("ConvFinQA:", CONVFINQA_DIR)

FinQA: d:\ai_project\FinGuide-AI\data\raw\FinQA\dataset
FinQA-30K: d:\ai_project\FinGuide-AI\data\raw\FinQA-30K
ConvFinQA: d:\ai_project\FinGuide-AI\data\raw\ConvFinQA


In [6]:
###### LOADING FINQA JSON DATA #################

with open(finqa_dataset_path / "train.json", "r", encoding="utf-8") as f:
    finqa_train = json.load(f)

with open(finqa_dataset_path / "dev.json", "r", encoding="utf-8") as f:
    finqa_dev = json.load(f)

with open(finqa_dataset_path / "test.json", "r", encoding="utf-8") as f:
    finqa_test = json.load(f)

print("FinQA")
print("Train:", len(finqa_train))
print("Dev:", len(finqa_dev))
print("Test:", len(finqa_test))

FinQA
Train: 6251
Dev: 883
Test: 1147


In [ ]:
###### LOADING FINQA-30K DATA #################

finqa_30k_data = []

for path in FINQA_30K_DIR.rglob("*.json"):
    if ".cache" in path.parts:
        continue

    with open(path, "r", encoding="utf-8") as f:
        records = json.load(f)

    domain = path.parent.name

    for record in records:
        record["_domain"] = domain
        finqa_30k_data.append(record)

print("FinQA-30K:", len(finqa_30k_data))

FinQA-30K: 34563


In [9]:
finqa_30k_data[0]

{'source_pdf': '09_chapter 1.pdf',
 'chunk_id': 0,
 'question': 'What are the three major segments that make up the Indian financial sector?',
 'answer': 'The three major segments are Banking Institutions, Development Financial Institutions, and Non-Banking Financial Intermediaries.',
 'type': 'factual',
 '_domain': 'Banking'}

In [10]:
###### LOADING CONVFINQA DATA #################
convfinqa_train = pd.read_parquet(
    CONVFINQA_DIR / "data" / "train-00000-of-00001.parquet"
)

convfinqa_test = pd.read_parquet(
    CONVFINQA_DIR / "data" / "test-00000-of-00001.parquet"
)

print("ConvFinQA")
print("Train:", len(convfinqa_train))
print("Test:", len(convfinqa_test))

ConvFinQA
Train: 3037
Test: 200


In [12]:
def convert_finqa_record(record):
    qa = record["qa"]

    context = "\n".join(
        record.get("pre_text", [])
        + record.get("post_text", [])
    )

    return {
        "id": f"FinQA_{record['id']}",
        "question": qa.get("question", "").strip(),
        "answer": qa.get("answer", "").strip(),
        "context": context.strip(),
        "table": record.get("table", []),
        "domain": "",
        "question_type": "",
        "reasoning": qa.get("program", ""),
        "source_dataset": "FinQA",
        "source_id": record["id"],
    }

In [13]:
finqa_unified_sample = convert_finqa_record(finqa_train[0])

finqa_unified_sample

{'id': 'FinQA_ADI/2009/page_49.pdf-1',
 'question': 'what is the the interest expense in 2009?',
 'answer': '380',
 'context': 'interest rate to a variable interest rate based on the three-month libor plus 2.05% ( 2.05 % ) ( 2.34% ( 2.34 % ) as of october 31 , 2009 ) .\nif libor changes by 100 basis points , our annual interest expense would change by $ 3.8 million .\nforeign currency exposure as more fully described in note 2i .\nin the notes to consolidated financial statements contained in item 8 of this annual report on form 10-k , we regularly hedge our non-u.s .\ndollar-based exposures by entering into forward foreign currency exchange contracts .\nthe terms of these contracts are for periods matching the duration of the underlying exposure and generally range from one month to twelve months .\ncurrently , our largest foreign currency exposure is the euro , primarily because our european operations have the highest proportion of our local currency denominated expenses .\nrelative

In [14]:
finqa_unified = (
    [convert_finqa_record(x) for x in finqa_train]
    + [convert_finqa_record(x) for x in finqa_dev]
    + [convert_finqa_record(x) for x in finqa_test]
)

print("Unified FinQA records:", len(finqa_unified))

Unified FinQA records: 8281


In [17]:
finqa_unified[1]

{'id': 'FinQA_ABMD/2012/page_75.pdf-1',
 'question': 'during the 2012 year , did the equity awards in which the prescribed performance milestones were achieved exceed the equity award compensation expense for equity granted during the year?',
 'answer': '',
 'context': 'abiomed , inc .\nand subsidiaries notes to consolidated financial statements 2014 ( continued ) note 8 .\nstock award plans and stock-based compensation ( continued ) restricted stock and restricted stock units the following table summarizes restricted stock and restricted stock unit activity for the fiscal year ended march 31 , 2012 : number of shares ( in thousands ) weighted average grant date fair value ( per share ) .\nthe remaining unrecognized compensation expense for outstanding restricted stock and restricted stock units , including performance-based awards , as of march 31 , 2012 was $ 7.1 million and the weighted-average period over which this cost will be recognized is 2.2 years .\nthe weighted average grant

In [18]:
def convert_finqa_30k_record(record):
    return {
        "id": f"FinQA-30K_{record['source_pdf']}_{record['chunk_id']}",
        "question": record.get("question", "").strip(),
        "answer": record.get("answer", "").strip(),
        "context": "",
        "table": "",
        "domain": record.get("_domain", ""),
        "question_type": record.get("type", ""),
        "reasoning": "",
        "source_dataset": "FinQA-30K",
        "source_id": f"{record['source_pdf']}_{record['chunk_id']}",
    }

In [19]:
finqa_30k_unified = [
    convert_finqa_30k_record(record)
    for record in finqa_30k_data
]

print("Unified FinQA-30K records:", len(finqa_30k_unified))

Unified FinQA-30K records: 34563


In [20]:
print(finqa_30k_unified[0])

{'id': 'FinQA-30K_09_chapter 1.pdf_0', 'question': 'What are the three major segments that make up the Indian financial sector?', 'answer': 'The three major segments are Banking Institutions, Development Financial Institutions, and Non-Banking Financial Intermediaries.', 'context': '', 'table': '', 'domain': 'Banking', 'question_type': 'factual', 'reasoning': '', 'source_dataset': 'FinQA-30K', 'source_id': '09_chapter 1.pdf_0'}


In [21]:
def convert_convfinqa_record(record):
    context = "\n".join(
        list(record.get("pre_text", []))
        + list(record.get("post_text", []))
    )

    return {
        "id": f"ConvFinQA_{record['id']}",
        "question": str(record.get("question", "")).strip(),
        "answer": str(record.get("answer", "")).strip(),
        "context": context.strip(),
        "table": record.get("table", "").tolist()
                if hasattr(record.get("table"), "tolist")
                else record.get("table", ""),
        "domain": "",
        "question_type": "conversational",
        "reasoning": record.get("steps", "").tolist()
                    if hasattr(record.get("steps"), "tolist")
                    else record.get("steps", ""),
        "source_dataset": "ConvFinQA",
        "source_id": str(record["id"]),
    }

In [22]:
convfinqa_unified = [
    convert_convfinqa_record(row)
    for _, row in pd.concat(
        [convfinqa_train, convfinqa_test],
        ignore_index=True
    ).iterrows()
]

print("Unified ConvFinQA records:", len(convfinqa_unified))

Unified ConvFinQA records: 3237


In [23]:
print(convfinqa_unified[0])

{'id': 'ConvFinQA_Single_JKHY/2009/page_28.pdf-3', 'question': 'what was the percentage change in the net cash from operating activities from 2008 to 2009', 'answer': '14.1%', 'context': '26 | 2009 annual report in fiscal 2008 , revenues in the credit union systems and services business segment increased 14% ( 14 % ) from fiscal 2007 .\nall revenue components within the segment experienced growth during fiscal 2008 .\nlicense revenue generated the largest dollar growth in revenue as episys ae , our flagship core processing system aimed at larger credit unions , experienced strong sales throughout the year .\nsupport and service revenue , which is the largest component of total revenues for the credit union segment , experienced 34 percent growth in eft support and 10 percent growth in in-house support .\ngross profit in this business segment increased $ 9344 in fiscal 2008 compared to fiscal 2007 , due primarily to the increase in license revenue , which carries the highest margins .\n

In [24]:
convfinqa_train.columns

Index(['pre_text', 'post_text', 'filename', 'table_ori', 'table', 'question',
       'answer', 'steps', 'id'],
      dtype='str')

In [26]:
convfinqa_train.head()

,pre_text,post_text,filename,table_ori,table,question,answer,steps,id
0,"[26 | 2009 annual report in fiscal 2008 , reve...","[year ended june 30 , cash provided by operati...",JKHY/2009/page_28.pdf,"[[, Year ended June 30, 2009], [2008, 2007], [...","[[2008, year ended june 30 2009 2008, year end...",what was the percentage change in the net cash...,14.1%,"[{'arg1': '206588', 'arg2': '181001', 'op': 'm...",Single_JKHY/2009/page_28.pdf-3
1,[substantially all of the goodwill and other i...,[the above unaudited pro forma financial infor...,RSG/2008/page_114.pdf,"[[, Year Ended December 31, 2008 (Unaudited), ...","[[, year ended december 31 2008 ( unaudited ),...",what was the percent of the growth in the reve...,1.3%,"[{'arg1': '9362.2', 'arg2': '9244.9', 'op': 'm...",Single_RSG/2008/page_114.pdf-2
2,[in a new business model such as the retail se...,[.],AAPL/2002/page_23.pdf,"[[, 2002, 2001, 2000], [Net sales, $5,742, $5,...","[[, 2002, 2001, 2000], [net sales, $ 5742, $ 5...",what was the percentage change in net sales fr...,-32%,"[{'arg1': '5363', 'arg2': '7983', 'op': 'minus...",Single_AAPL/2002/page_23.pdf-1
3,[( 1 ) includes shares repurchased through our...,[.],UPS/2009/page_33.pdf,"[[, 12/31/04, 12/31/05, 12/31/06, 12/31/07, 12...","[[, 12/31/04, 12/31/05, 12/31/06, 12/31/07, 12...",what was the difference in percentage cumulati...,-26.16%,"[{'arg1': '75.95', 'arg2': 'const_100', 'op': ...",Single_UPS/2009/page_33.pdf-2
4,[( 1 ) includes shares repurchased through our...,[.],UPS/2009/page_33.pdf,"[[, 12/31/04, 12/31/05, 12/31/06, 12/31/07, 12...","[[, 12/31/04, 12/31/05, 12/31/06, 12/31/07, 12...",,,[],Double_UPS/2009/page_33.pdf


In [27]:
unified_data = (
    finqa_unified
    + finqa_30k_unified
    + convfinqa_unified
)

print("Total unified records:", len(unified_data))

Total unified records: 46081


In [28]:
print(unified_data[0])
print(unified_data[8281])
print(unified_data[-1])

{'id': 'FinQA_ADI/2009/page_49.pdf-1', 'question': 'what is the the interest expense in 2009?', 'answer': '380', 'context': 'interest rate to a variable interest rate based on the three-month libor plus 2.05% ( 2.05 % ) ( 2.34% ( 2.34 % ) as of october 31 , 2009 ) .\nif libor changes by 100 basis points , our annual interest expense would change by $ 3.8 million .\nforeign currency exposure as more fully described in note 2i .\nin the notes to consolidated financial statements contained in item 8 of this annual report on form 10-k , we regularly hedge our non-u.s .\ndollar-based exposures by entering into forward foreign currency exchange contracts .\nthe terms of these contracts are for periods matching the duration of the underlying exposure and generally range from one month to twelve months .\ncurrently , our largest foreign currency exposure is the euro , primarily because our european operations have the highest proportion of our local currency denominated expenses .\nrelative to

In [36]:
count = 0
for data in unified_data:
    if data['question'] == "" or data['answer'] == "" :
        count += 1

print("count :", count)

count : 1068


In [37]:
from collections import Counter

empty_by_source = Counter()

for data in unified_data:
    if data["question"].strip() == "" or data["answer"].strip() == "":
        empty_by_source[data["source_dataset"]] += 1

print(empty_by_source)

Counter({'ConvFinQA': 994, 'FinQA': 74})


In [38]:
cleaned_data = [
    record
    for record in unified_data
    if record["question"].strip() and record["answer"].strip()
]

print("Before:", len(unified_data))
print("After :", len(cleaned_data))
print("Removed:", len(unified_data) - len(cleaned_data))

Before: 46081
After : 45013
Removed: 1068


In [39]:
## have Deduplication ?

from collections import Counter

qa_pairs = Counter(
    (record["question"].strip().lower(),
     record["answer"].strip().lower())
    for record in cleaned_data
)

duplicate_records = sum(
    count - 1
    for count in qa_pairs.values()
    if count > 1
)

duplicate_groups = sum(
    1
    for count in qa_pairs.values()
    if count > 1
)

print("Unique QA pairs:", len(qa_pairs))
print("Duplicate groups:", duplicate_groups)
print("Duplicate records:", duplicate_records)

Unique QA pairs: 42661
Duplicate groups: 2148
Duplicate records: 2352


In [41]:
cleaned_data

[{'id': 'FinQA_ADI/2009/page_49.pdf-1',
  'question': 'what is the the interest expense in 2009?',
  'answer': '380',
  'context': 'interest rate to a variable interest rate based on the three-month libor plus 2.05% ( 2.05 % ) ( 2.34% ( 2.34 % ) as of october 31 , 2009 ) .\nif libor changes by 100 basis points , our annual interest expense would change by $ 3.8 million .\nforeign currency exposure as more fully described in note 2i .\nin the notes to consolidated financial statements contained in item 8 of this annual report on form 10-k , we regularly hedge our non-u.s .\ndollar-based exposures by entering into forward foreign currency exchange contracts .\nthe terms of these contracts are for periods matching the duration of the underlying exposure and generally range from one month to twelve months .\ncurrently , our largest foreign currency exposure is the euro , primarily because our european operations have the highest proportion of our local currency denominated expenses .\nrela

In [42]:
duplicate_groups_data = {}

for record in cleaned_data:
    key = (
        record["question"].strip().lower(),
        record["answer"].strip().lower()
    )

    duplicate_groups_data.setdefault(key, []).append(record)

cross_source_duplicates = 0

for records in duplicate_groups_data.values():
    sources = {record["source_dataset"] for record in records}

    if len(sources) > 1:
        cross_source_duplicates += 1

print("Cross-dataset duplicate groups:", cross_source_duplicates)

Cross-dataset duplicate groups: 2071


In [43]:
def normalize_text(text):
    return " ".join(str(text).lower().split())


def information_score(record):
    score = 0

    if record["context"]:
        score += len(record["context"])

    if record["table"]:
        score += len(str(record["table"]))

    if record["reasoning"]:
        score += len(str(record["reasoning"]))

    if record["domain"]:
        score += 10

    if record["question_type"]:
        score += 10

    return score


deduplicated_map = {}

for record in cleaned_data:
    key = (
        normalize_text(record["question"]),
        normalize_text(record["answer"])
    )

    if key not in deduplicated_map:
        deduplicated_map[key] = record
    else:
        current = deduplicated_map[key]

        if information_score(record) > information_score(current):
            deduplicated_map[key] = record


deduplicated_data = list(deduplicated_map.values())

print("Before deduplication:", len(cleaned_data))
print("After deduplication :", len(deduplicated_data))
print("Removed duplicates  :", len(cleaned_data) - len(deduplicated_data))

Before deduplication: 45013
After deduplication : 42661
Removed duplicates  : 2352


In [44]:
quality_issues = {
    "very_short_question": 0,
    "very_short_answer": 0,
    "very_long_context": 0,
    "missing_source": 0,
}

for record in deduplicated_data:

    question = record["question"].strip()
    answer = record["answer"].strip()
    context = record["context"].strip()

    if len(question) < 10:
        quality_issues["very_short_question"] += 1

    if len(answer) < 2:
        quality_issues["very_short_answer"] += 1

    if len(context) > 20000:
        quality_issues["very_long_context"] += 1

    if not record["source_dataset"].strip():
        quality_issues["missing_source"] += 1

print(quality_issues)

{'very_short_question': 0, 'very_short_answer': 94, 'very_long_context': 0, 'missing_source': 0}


In [46]:
quality_report = {
    "empty_question": 0,
    "empty_answer": 0,
    "missing_id": 0,
    "missing_source_dataset": 0,
    "duplicate_ids": 0,
}

seen_ids = set()

for record in deduplicated_data:
    question = str(record.get("question", "")).strip()
    answer = str(record.get("answer", "")).strip()
    record_id = str(record.get("id", "")).strip()
    source = str(record.get("source_dataset", "")).strip()

    if not question:
        quality_report["empty_question"] += 1

    if not answer:
        quality_report["empty_answer"] += 1

    if not record_id:
        quality_report["missing_id"] += 1

    if not source:
        quality_report["missing_source_dataset"] += 1

    if record_id in seen_ids:
        quality_report["duplicate_ids"] += 1
    else:
        seen_ids.add(record_id)

quality_report

{'empty_question': 0,
 'empty_answer': 0,
 'missing_id': 0,
 'missing_source_dataset': 0,
 'duplicate_ids': 31329}

In [49]:

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

output_path = PROCESSED_DATA_DIR / "finGuide_deduplicated.json"


def make_json_serializable(obj):
    """Convert NumPy/Pandas objects into JSON-compatible Python objects."""

    if isinstance(obj, np.ndarray):
        return obj.tolist()

    if isinstance(obj, np.generic):
        return obj.item()

    if isinstance(obj, dict):
        return {
            key: make_json_serializable(value)
            for key, value in obj.items()
        }

    if isinstance(obj, list):
        return [
            make_json_serializable(value)
            for value in obj
        ]

    if isinstance(obj, tuple):
        return [
            make_json_serializable(value)
            for value in obj
        ]

    return obj


# Convert dataset into JSON-compatible structure
json_ready_data = make_json_serializable(deduplicated_data)


# Save
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(
        json_ready_data,
        f,
        ensure_ascii=False,
        indent=2
    )


print(f"Saved: {output_path}")
print(f"Records: {len(json_ready_data):,}")
print(f"File size: {output_path.stat().st_size / (1024 * 1024):.2f} MB")

Saved: d:\ai_project\FinGuide-AI\data\processed\finGuide_deduplicated.json
Records: 42,661
File size: 59.23 MB


In [50]:
with open(output_path, "r", encoding="utf-8") as f:
    saved_data = json.load(f)

print("Reload successful")
print("Records:", len(saved_data))
print("First record:")
print(json.dumps(saved_data[0], indent=2, ensure_ascii=False))

Reload successful
Records: 42661
First record:
{
  "id": "FinQA_ADI/2009/page_49.pdf-1",
  "question": "what is the the interest expense in 2009?",
  "answer": "380",
  "context": "interest rate to a variable interest rate based on the three-month libor plus 2.05% ( 2.05 % ) ( 2.34% ( 2.34 % ) as of october 31 , 2009 ) .\nif libor changes by 100 basis points , our annual interest expense would change by $ 3.8 million .\nforeign currency exposure as more fully described in note 2i .\nin the notes to consolidated financial statements contained in item 8 of this annual report on form 10-k , we regularly hedge our non-u.s .\ndollar-based exposures by entering into forward foreign currency exchange contracts .\nthe terms of these contracts are for periods matching the duration of the underlying exposure and generally range from one month to twelve months .\ncurrently , our largest foreign currency exposure is the euro , primarily because our european operations have the highest proportion o

In [56]:
deduplicated_data[0]

{'id': 'FinQA_ADI/2009/page_49.pdf-1',
 'question': 'what is the the interest expense in 2009?',
 'answer': '380',
 'context': 'interest rate to a variable interest rate based on the three-month libor plus 2.05% ( 2.05 % ) ( 2.34% ( 2.34 % ) as of october 31 , 2009 ) .\nif libor changes by 100 basis points , our annual interest expense would change by $ 3.8 million .\nforeign currency exposure as more fully described in note 2i .\nin the notes to consolidated financial statements contained in item 8 of this annual report on form 10-k , we regularly hedge our non-u.s .\ndollar-based exposures by entering into forward foreign currency exchange contracts .\nthe terms of these contracts are for periods matching the duration of the underlying exposure and generally range from one month to twelve months .\ncurrently , our largest foreign currency exposure is the euro , primarily because our european operations have the highest proportion of our local currency denominated expenses .\nrelative

In [58]:
empty_keys = [
    k
    for record in deduplicated_data 
    for k, v in record.items()
    if v is None
    or (isinstance(v, str) and v.strip() == '')
    or (isinstance(v, (list, dict)) and len(v) == 0)
]

print("Empty keys in this record:", empty_keys)

Empty keys in this record: ['domain', 'question_type', 'domain', 'question_type', 'domain', 'question_type', 'domain', 'domain', 'question_type', 'domain', 'domain', 'question_type', 'domain', 'domain', 'question_type', 'domain', 'question_type', 'domain', 'question_type', 'domain', 'domain', 'question_type', 'domain', 'question_type', 'domain', 'question_type', 'domain', 'question_type', 'domain', 'question_type', 'domain', 'question_type', 'domain', 'question_type', 'domain', 'domain', 'question_type', 'domain', 'question_type', 'domain', 'question_type', 'domain', 'question_type', 'domain', 'question_type', 'domain', 'question_type', 'domain', 'question_type', 'domain', 'domain', 'domain', 'question_type', 'domain', 'domain', 'question_type', 'domain', 'domain', 'question_type', 'domain', 'domain', 'domain', 'question_type', 'domain', 'domain', 'question_type', 'domain', 'question_type', 'domain', 'question_type', 'domain', 'question_type', 'domain', 'question_type', 'domain', 'ques

In [60]:
dick= dict()
for key in empty_keys:
    dick[key] = dick.get(key, 0) + 1

print(dick) 

{'domain': 8123, 'question_type': 6054, 'context': 34538, 'table': 34538, 'reasoning': 34538}
